In [1]:
# Importing the requests library to handle HTTP requests
import requests

# Defining the URL of the FPL API's general information endpoint
url = 'https://fantasy.premierleague.com/api/bootstrap-static/'

# Sending a GET request to the FPL API and storing the response
response = requests.get(url)

# Checking if the request was successful (status code 200 indicates success)
if response.status_code == 200:
    # Parsing the JSON response content into a Python dictionary
    data = response.json()
    print("Data downloaded successfully!")
else:
    # If the request was not successful, print an error message
    print(f"Failed to retrieve data. Status code: {response.status_code}")

# Displaying the keys of the downloaded data to understand its structure
print(data.keys())

# Example: Accessing specific data (e.g., players' information)
players = data['elements']
print(f"Number of players data downloaded: {len(players)}")

# Displaying the first player's data for reference
print(players[0])


Data downloaded successfully!
dict_keys(['events', 'game_settings', 'phases', 'teams', 'total_players', 'elements', 'element_stats', 'element_types'])
Number of players data downloaded: 591
{'chance_of_playing_next_round': 75, 'chance_of_playing_this_round': None, 'code': 438098, 'cost_change_event': 0, 'cost_change_event_fall': 0, 'cost_change_start': 0, 'cost_change_start_fall': 0, 'dreamteam_count': 0, 'element_type': 3, 'ep_next': '2.3', 'ep_this': None, 'event_points': 0, 'first_name': 'Fábio', 'form': '0.0', 'id': 1, 'in_dreamteam': False, 'news': 'Hip injury - 75% chance of playing', 'news_added': '2024-08-12T17:00:05.432893Z', 'now_cost': 55, 'photo': '438098.jpg', 'points_per_game': '2.2', 'second_name': 'Ferreira Vieira', 'selected_by_percent': '0.1', 'special': False, 'squad_number': None, 'status': 'd', 'team': 1, 'team_code': 3, 'total_points': 24, 'transfers_in': 0, 'transfers_in_event': 0, 'transfers_out': 0, 'transfers_out_event': 0, 'value_form': '0.0', 'value_season':

In [2]:
# Import pandas for data manipulation
import pandas as pd

# Convert the player data into a DataFrame
df_players = pd.DataFrame(players)

# Save the DataFrame to a CSV file
df_players.to_csv('fpl_players.csv', index=False)

# Optionally, save to an Excel file
# df_players.to_excel('fpl_players.xlsx', index=False)

print("Player data saved to fpl_players.csv and fpl_players.xlsx")


Player data saved to fpl_players.csv and fpl_players.xlsx


In [3]:
import itertools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re
import requests
import seaborn as sns
import warnings

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    r2_score,
    mean_absolute_percentage_error,
    mean_squared_error
)
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder
)

# Disable some annoying & confusing warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
pd.options.mode.chained_assignment = None

In [4]:
data = pd.read_csv("/kaggle/input/fantasy-football/fantasy_football.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/fantasy-football/fantasy_football.csv'

In [6]:
# Set to True to get the latest data direct from GitHub
# This might fail if your notebook isn't connected to the Internet,
# or if you're not logged in to a verified account.
# Most people should be fine using the Kaggle version of the data
# and can just leave this set to False.
get_online_data = False

if get_online_data:
    try:
        data = []

        # Get historic data from GitHub
        for year in ["2016-17", "2017-18", "2018-19", "2019-20", "2020-21", "2021-22", "2022-23", "2023-24"]:
            df = pd.read_csv(f"https://github.com/vaastav/Fantasy-Premier-League/raw/master/data/{year}/players_raw.csv")
            df["season_x"] = year
            data.append(df)

        # Stack all the rows together
        data = pd.concat(data)
        # Pick the columns we want
        # We'll do feature selection later, don't touch this list
        # (If you want a feature from the source I've missed, you can add it
        # but you might need to force a refresh by editing the top of this cell)
        data = data[[
            # Indices
            'season_x', 'first_name', 'second_name',
            # Features
            'assists', 'bonus', 'bps', 'chance_of_playing_next_round',
            'chance_of_playing_this_round', 'clean_sheets',
            'creativity', 'dreamteam_count', 'ea_index',
            'element_type',
            'form', 'goals_conceded', 'goals_scored', 'ict_index',
            'in_dreamteam', 'influence', 'loaned_in', 'loaned_out', 'loans_in',
            'loans_out', 'minutes', 'now_cost', 'own_goals',
            'penalties_missed', 'penalties_saved', 'points_per_game',
            'red_cards', 'saves', 'selected_by_percent', 'special',
            'status',
            'team',
            'threat',
            'transfers_in', 'transfers_out',
            'value_form', 'value_season',
            'yellow_cards', 'creativity_rank',
            'creativity_rank_type', 'ict_index_rank', 'ict_index_rank_type',
            'influence_rank', 'influence_rank_type', 'threat_rank',
            'threat_rank_type', 'corners_and_indirect_freekicks_order',
            'direct_freekicks_order',
            'penalties_order',
            'clean_sheets_per_90', 'expected_assists', 'expected_assists_per_90',
            'expected_goal_involvements', 'expected_goal_involvements_per_90',
            'expected_goals', 'expected_goals_conceded',
            'expected_goals_conceded_per_90', 'expected_goals_per_90', 'form_rank',
            'form_rank_type', 'goals_conceded_per_90', 'now_cost_rank',
            'now_cost_rank_type', 'points_per_game_rank',
            'points_per_game_rank_type', 'saves_per_90', 'selected_rank',
            'selected_rank_type', 'starts', 'starts_per_90',
            # Target
            'total_points'
        ]]

        # Combine first name and last name into one column
        data["name"] = data['first_name'] + " " + data['second_name']
        data = data.drop(['first_name', 'second_name'], axis=1)
        
        # Get team mappings
        # Get mappings up to 2023-4
        teams = (
            pd.read_csv("https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/master_team_list.csv")
            .set_index(['season', 'team'])
        )
        # Get mappings for 2023-4
        teams_2023_4 = (
            pd.read_csv("https://raw.githubusercontent.com/vaastav/Fantasy-Premier-League/master/data/2023-24/teams.csv")
            [['id', 'name']]
        )
        teams_2023_4["season"] = "2023-24"
        teams_2023_4.columns = ["team", "team_name", "season"]
        teams_2023_4 = teams_2023_4.set_index(['season', 'team'])

        teams = pd.concat([teams, teams_2023_4])

        # Replace team numbers with team names in the original data
        data = data.join(teams, on=['season_x', 'team']).drop('team', axis=1)
        
        # Save
        data.to_csv("fantasy_football.csv")
    except:
        print("Failed to fetch data - are you definitely logged in?")

# Or just use the data I've already saved as a Kaggle dataset 
else:
    data = pd.read_csv("/kaggle/input/fantasy-football/fantasy_football.csv")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/fantasy-football/fantasy_football.csv'

In [7]:
data = pd.read_csv("fantasy_football.csv")

In [8]:
import requests
import pandas as pd

# Make API request to get gameweek 1 data
url = "https://fantasy.premierleague.com/api/event/1/live/"
response = requests.get(url)

# Check if request was successful
if response.status_code == 200:
    # Parse JSON data
    data = response.json()
    
    # Extract player data
    players = data['elements']
    
    # Convert to DataFrame
    df = pd.DataFrame(players)
    
    # Select relevant columns
    columns = ['id', 'stats.minutes', 'stats.goals_scored', 'stats.assists', 
               'stats.clean_sheets', 'stats.goals_conceded', 'stats.own_goals',
               'stats.penalties_saved', 'stats.penalties_missed', 'stats.yellow_cards',
               'stats.red_cards', 'stats.saves', 'stats.bonus', 'stats.bps', 'stats.total_points']
    
    df_selected = df[columns]
    
    # Rename columns for clarity
    df_selected.columns = ['player_id', 'minutes', 'goals', 'assists', 'clean_sheets',
                           'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed',
                           'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'total_points']
    
    print(df_selected.head())
else:
    print(f"Failed to retrieve data. Status code: {response.status_code}")

KeyError: "['stats.minutes', 'stats.goals_scored', 'stats.assists', 'stats.clean_sheets', 'stats.goals_conceded', 'stats.own_goals', 'stats.penalties_saved', 'stats.penalties_missed', 'stats.yellow_cards', 'stats.red_cards', 'stats.saves', 'stats.bonus', 'stats.bps', 'stats.total_points'] not in index"

In [9]:
import requests
import pandas as pd

# Make API request to get gameweek 1 data
url = "https://fantasy.premierleague.com/api/event/1/live/"
response = requests.get(url)

# Check if request was successful
if response.status_code == 200:
    # Parse JSON data
    data = response.json()
    
    # Extract player data
    players = data['elements']
    
    # Convert to DataFrame
    df = pd.DataFrame(players)
    
    # Print column names to see what's available
    print("Available columns:", df.columns.tolist())
    
    # Flatten the 'stats' dictionary into separate columns
    stats_df = pd.DataFrame([player['stats'] for player in players])
    
    # Combine the original DataFrame with the flattened stats
    df = pd.concat([df, stats_df], axis=1)
    
    # Select relevant columns
    columns = ['id', 'minutes', 'goals_scored', 'assists', 'clean_sheets',
               'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed',
               'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'total_points']
    
    df_selected = df[columns]
    
    # Rename columns for clarity
    df_selected.columns = ['player_id', 'minutes', 'goals', 'assists', 'clean_sheets',
                           'goals_conceded', 'own_goals', 'penalties_saved', 'penalties_missed',
                           'yellow_cards', 'red_cards', 'saves', 'bonus', 'bps', 'total_points']
    
    print(df_selected.head())
else:
    print(f"Failed to retrieve data. Status code: {response.status_code}")

Available columns: ['id', 'stats', 'explain']
   player_id  minutes  goals  assists  clean_sheets  goals_conceded  \
0          1        0      0        0             0               0   
1          2        5      0        0             0               0   
2          3       90      0        0             1               0   
3          4       90      1        1             1               0   
4          5        0      0        0             0               0   

   own_goals  penalties_saved  penalties_missed  yellow_cards  red_cards  \
0          0                0                 0             0          0   
1          0                0                 0             1          0   
2          0                0                 0             0          0   
3          0                0                 0             0          0   
4          0                0                 0             0          0   

   saves  bonus  bps  total_points  
0      0      0    0             

In [10]:

# List of player IDs you want to query
player_ids = [4, 17, 91, 116, 181, 199, 231, 311, 328, 350, 401]

# Filter the DataFrame to get data for the specified player IDs
df_filtered = df_selected[df_selected['player_id'].isin(player_ids)]

# Select relevant columns: player_id and total_points
df_filtered = df_filtered[['player_id', 'total_points']]

# Display the results
print(df_filtered)



     player_id  total_points
3            4            12
16          17            12
90          91             4
115        116             0
180        181             1
198        199             2
230        231             1
310        311             8
327        328            14
349        350             7
400        401             5
